# AI Facial Recognition — Compliance Intelligence Explorer
### 7CS525 Human & Legal Aspects of Cyber Security · University of Derby 2025/26 · Student: 100796311

**How to run:**
- Google Colab: `Runtime → Run All` — installs are automatic
- Local: `pip install pyvis plotly pandas networkx` then run all cells

| Module | Output |
|--------|--------|
| **1** | System Architecture Pipeline |
| **2** | EU AI Act Dual-Regime Analysis |
| **3** | Articles 9-15 Compliance Cards |
| **4** | EU vs UK Comparison Table |
| **5** | Accountability Scenario |
| **6** | Human-in-the-Loop Analysis |
| **7** | Stakeholder Map (Plotly) |
| **8** | Risk Register RAG + L x I Matrix |
| **9** | Compliance Knowledge Graph (pyvis) |
| **10** | Reflective Commentary |
| **11** | Executive Dashboard + KPIs |

In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
for pkg in ["plotly", "pandas", "networkx", "pyvis"]:
    try: __import__(pkg.replace("-","_"))
    except ImportError: install(pkg)
import warnings; warnings.filterwarnings("ignore")
import networkx as nx, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyvis.network import Network
from IPython.display import display, HTML
from datetime import datetime

C = {"bg":"#050B14","panel":"#0D1B2E","panel2":"#0A1628","text":"#E8F0FE","muted":"#64748B",
     "blue":"#2563EB","cyan":"#06B6D4","red":"#EF4444","amber":"#F59E0B",
     "green":"#22C55E","purple":"#8B5CF6","orange":"#F97316"}
STATUS_COL = {"GAP":C["red"],"PARTIAL":C["amber"],"MET":C["green"],"TRIGGER":C["blue"]}

SYSTEM = {"name":"AI Facial Recognition Access Control","sector":"UK Retail Banking",
          "employees":4500,"sites":3,"threshold":92,"fpr":0.025,
          "daily_events":50000,"analyst_capacity":500,
          "eu_class":"HIGH RISK","annex_ref":"Annex III Pt.1","deadline":"August 2026"}
FPD  = int(SYSTEM["daily_events"]*SYSTEM["fpr"])
UNREV= max(0,FPD-SYSTEM["analyst_capacity"])
PCT  = round(UNREV/FPD*100) if FPD else 0

ARTICLES = {
    "AnnexIII": {"label":"Annex III Pt.1","what":"HIGH RISK trigger — biometric ID systems","status":"TRIGGER","gap":"All Arts.9-17 obligations apply","group":"trigger"},
    "Art5":     {"label":"Art.5(1)(h)","what":"Potential BAN in publicly accessible spaces","status":"GAP","gap":"Bank lobby likely qualifies under Art.3(44) — legally unsettled","group":"trigger"},
    "Art_9":    {"label":"Art.9 Risk Mgmt","what":"Continuous documented risk management throughout AI lifecycle","status":"GAP","gap":"No documented continuous risk management; lifecycle risk register absent","group":"primary"},
    "Art_10":   {"label":"Art.10 Data Gov","what":"Training data representative, bias-examined, complete","status":"GAP","gap":"Training data not audited; NIST 2019: error rates 100x higher for certain groups","group":"primary"},
    "Art_11":   {"label":"Art.11 Tech Docs","what":"Full technical documentation before deployment","status":"GAP","gap":"Vendor documentation not shared with deployer bank","group":"primary"},
    "Art_12":   {"label":"Art.12 Logging","what":"Automatic logging of events throughout system lifetime","status":"PARTIAL","gap":"Transaction logs exist but decision-path attribution not preserved","group":"primary"},
    "Art_13":   {"label":"Art.13 Transparency","what":"System output interpretable; customers informed","status":"GAP","gap":"No XAI layer; CNN explainability gap; no customer-facing explanation","group":"primary"},
    "Art_14":   {"label":"Art.14 Human Oversight","what":"Operators must understand, intervene and interrupt","status":"GAP","gap":"Two-person verification rule not implemented — CRITICAL GAP","group":"primary"},
    "Art_15":   {"label":"Art.15 Robustness","what":"Resilient against errors, drift and adversarial attacks","status":"GAP","gap":"Drift monitoring absent; adversarial robustness untested","group":"primary"},
    "Art_16":   {"label":"Art.16 Provider Duties","what":"Vendor must ensure Arts.9-15 compliance","status":"GAP","gap":"Vendor NOT contractually required to comply — Deployer Trap","group":"primary"},
    "Art_26":   {"label":"Art.26 Deployer Duties","what":"Bank must ensure oversight, monitor, inform affected persons","status":"GAP","gap":"Override rates not monitored; affected persons not systematically informed","group":"accountability"},
    "GDPR_22":  {"label":"UK GDPR Art.22","what":"Right not to be subject to solely automated decisions","status":"GAP","gap":"Art.22(3) review right not implemented for false-rejection victims","group":"parallel"},
    "EqAct":    {"label":"Equality Act s.19","what":"Prohibition on indirect discrimination via proxy variables","status":"GAP","gap":"No bias audit conducted; NIST 2019 documents 4x higher error rate for Black women","group":"parallel"},
    "FCA_CD":   {"label":"FCA Consumer Duty","what":"Deliver good outcomes for retail customers","status":"PARTIAL","gap":"False positives cause foreseeable customer harm without accessible remedy","group":"parallel"},
    "RRO":      {"label":"RRO 2005","what":"Responsible person must ensure routes to safety are clear","status":"GAP","gap":"Doors must unlock on fire alarm — integration not tested","group":"parallel"},
}
EDGES = [
    ("AnnexIII","Art_9","TRIGGERS",C["red"],3.0),("AnnexIII","Art_10","TRIGGERS",C["red"],2.5),
    ("AnnexIII","Art_11","TRIGGERS",C["red"],2.5),("AnnexIII","Art_12","TRIGGERS",C["red"],2.5),
    ("AnnexIII","Art_13","TRIGGERS",C["red"],2.5),("AnnexIII","Art_14","TRIGGERS",C["red"],3.5),
    ("AnnexIII","Art_15","TRIGGERS",C["red"],2.5),("AnnexIII","Art_16","TRIGGERS",C["red"],2.5),
    ("AnnexIII","Art_26","TRIGGERS",C["red"],2.5),
    ("Art5","AnnexIII","ALONGSIDE",C["red"],2.0),
    ("Art_9","Art_10","REQUIRES",C["blue"],2.0),("Art_9","Art_12","REQUIRES",C["blue"],2.0),
    ("Art_10","Art_15","SUPPORTS",C["green"],1.5),("Art_10","EqAct","ALIGNS WITH",C["green"],1.5),
    ("Art_11","Art_12","REFERENCES",C["muted"],1.5),("Art_13","Art_14","ENABLES",C["green"],2.0),
    ("Art_13","GDPR_22","ALIGNS WITH",C["cyan"],1.5),("Art_14","Art_9","INFORMS",C["amber"],1.5),
    ("Art_14","GDPR_22","ALIGNS WITH",C["cyan"],1.5),("Art_15","Art_9","INFORMS",C["amber"],1.5),
    ("Art_16","Art_9","RESPONSIBLE",C["purple"],2.0),("Art_16","Art_10","RESPONSIBLE",C["purple"],2.0),
    ("Art_16","Art_11","RESPONSIBLE",C["purple"],2.0),("Art_26","Art_14","IMPLEMENTS",C["orange"],2.0),
    ("Art_26","Art_13","IMPLEMENTS",C["orange"],1.5),("Art_26","FCA_CD","ALIGNS WITH",C["green"],1.5),
    ("RRO","Art_14","REQUIRES",C["cyan"],1.5),
]

RISKS = pd.DataFrame([
    {"ID":"R-01","Risk":"False positive: legitimate face rejected without review","L":4,"I":5,
     "L_why":"2.5% FPR x 50,000 events = 1,250 daily wrongful blocks — structural","I_why":"Consumer Duty breach + Art.22 right denied + financial harm at scale","Reg":"EU AI Act Art.14; UK GDPR Art.22; FCA Consumer Duty","Mit":"Human review threshold; 24hr appeal SLA"},
    {"ID":"R-02","Risk":"Discriminatory scoring via proxy variables (indirect discrimination)","L":3,"I":5,
     "L_why":"NIST 2019: 100x higher error rates documented — foreseeable","I_why":"Equality Act s.19 violation; class action; EHRC enforcement","Reg":"Equality Act s.19; EU AI Act Art.10","Mit":"Quarterly bias audit; demographic parity testing"},
    {"ID":"R-03","Risk":"No meaningful explanation for rejected face — CNN opacity","L":5,"I":3,
     "L_why":"Every automated rejection creates unmet transparency obligation","I_why":"ICO enforcement precedent (cf. Experian 2020)","Reg":"EU AI Act Art.13; UK GDPR Arts.13-14","Mit":"SHAP/Grad-CAM explanation module; customer decision letter"},
    {"ID":"R-04","Risk":"Building safety conflict: doors lock in fire evacuation","L":3,"I":5,
     "L_why":"Integration with fire alarm system not tested or documented","I_why":"Unlimited fines; criminal prosecution of Senior Manager (RRO 2005 Art.32)","Reg":"RRO 2005 Arts.4,32; Building Regs 2010","Mit":"Automatic door release on alarm; test in every fire drill"},
    {"ID":"R-05","Risk":"Automation bias — guards rubber-stamp AI rejections","L":4,"I":4,
     "L_why":"Inherent in high-volume time-pressured lobby workflow — structural","I_why":"Nullifies Art.14(5) compliance — human oversight in form only","Reg":"EU AI Act Art.14(5)","Mit":"Two-person verification; cognitive forcing; override KPIs"},
    {"ID":"R-06","Risk":"Vendor opaque model — Deployer Trap (Art.16/Art.26)","L":4,"I":4,
     "L_why":"Standard black-box-as-a-service commercial model","I_why":"Bank bears Art.26 liability it cannot independently discharge","Reg":"EU AI Act Arts.16, 26","Mit":"Contractual XAI + right-to-audit + Art.16 compliance clause"},
])
RISKS["Score"] = RISKS["L"]*RISKS["I"]
RISKS["Severity"] = RISKS["Score"].apply(lambda s:"Critical" if s>=16 else "High" if s>=12 else "Medium" if s>=6 else "Low")
SEV_COL = {"Critical":C["red"],"High":C["orange"],"Medium":C["amber"],"Low":C["green"]}
RISKS["SevCol"] = RISKS["Severity"].map(SEV_COL)

non_trigger = [k for k,v in ARTICLES.items() if v["status"] not in ("TRIGGER",)]
gaps_n    = sum(1 for k in non_trigger if ARTICLES[k]["status"]=="GAP")
partial_n = sum(1 for k in non_trigger if ARTICLES[k]["status"]=="PARTIAL")
met_n     = sum(1 for k in non_trigger if ARTICLES[k]["status"]=="MET")

def dark_layout(title="",h=520):
    return dict(paper_bgcolor=C["bg"],plot_bgcolor=C["panel"],
                font=dict(color=C["text"],family="monospace"),
                margin=dict(l=20,r=20,t=60,b=20),
                hoverlabel=dict(bgcolor="#0F2040",bordercolor=C["blue"],font=dict(color=C["text"],size=11)),
                title=dict(text=title,font=dict(size=14,color=C["text"])),height=h,showlegend=False)

display(HTML(f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.4);border-radius:12px;padding:16px;font-family:monospace;color:{C["text"]};font-size:12px"><b style="color:{C["cyan"]}">DATA LOADED</b><br>System: {SYSTEM["name"]}<br>Classification: <b style="color:{C["red"]}">{SYSTEM["eu_class"]} — {SYSTEM["annex_ref"]}</b><br>Events/day: {SYSTEM["daily_events"]:,} | FP/day: {FPD:,} | Unreviewed: <b style="color:{C["red"]}">{UNREV:,} ({PCT}%)</b></div>'))
print("All libraries loaded.")

In [ ]:
# ── Module 1: System Architecture ───────────────────────────────────────────
stages=["Data Capture\n(Cameras/Anti-spoof)","Feature Extraction\n(128-D Embedding)",
        "Matching Engine\n(Threshold 92%)","Decision Layer\n(Score routing)",
        "Human Review\n(Officer/Reception)"]
stage_cols=[C["cyan"],C["blue"],C["purple"],C["orange"],C["green"]]
fig=go.Figure()
for i,(s,col) in enumerate(zip(stages,stage_cols)):
    fig.add_trace(go.Scatter(x=[i],y=[1],mode="markers+text",
        marker=dict(size=82,color=col,opacity=0.85,line=dict(color="white",width=2)),
        text=[s],textposition="bottom center",textfont=dict(size=9,color="white"),
        hovertemplate=f"<b>{s}</b><extra>Stage {i+1}</extra>"))
    if i<len(stages)-1:
        fig.add_annotation(x=i+0.5,y=1,text="▶",font=dict(size=24,color=C["muted"]),showarrow=False)
stakeholders=[("SOC Officers\nReception",0.5,0.15,C["cyan"]),
              ("CISO\nAI Governance",1.8,0.15,C["purple"]),
              ("DPO\nGDPR Art.22",2.8,0.15,C["amber"]),
              ("AI Vendor\nArt.16 Duties",3.5,0.15,C["orange"]),
              ("Board/SM&CR\nNIS2 Art.20",2.0,0.02,C["green"])]
for name,x,y,col in stakeholders:
    fig.add_trace(go.Scatter(x=[x],y=[y],mode="markers+text",
        marker=dict(size=42,color=col,opacity=0.75,symbol="diamond",line=dict(color="white",width=1.5)),
        text=[name],textposition="bottom center",textfont=dict(size=8,color="white"),
        hovertemplate=f"<b>{name}</b><extra>Stakeholder</extra>"))
layout=dark_layout("AI Facial Recognition — Processing Pipeline & Human Touchpoints",h=420)
layout.update(xaxis=dict(showticklabels=False,showgrid=False,zeroline=False),
              yaxis=dict(showticklabels=False,showgrid=False,zeroline=False,range=[-0.15,1.55]))
fig.update_layout(**layout); fig.show()

In [ ]:
# ── Module 2: EU AI Act Dual-Regime ─────────────────────────────────────────
tiers=[("UNACCEPTABLE",C["red"],"Art.5(1)(h): Potential BAN",
        "Real-time biometric ID in publicly accessible spaces is PROHIBITED. Art.3(44) includes any space accessible to an undetermined number of people. Bank lobby with walk-in visitors almost certainly qualifies.",
        "POSSIBLY ENGAGED — lobby status legally unsettled"),
       ("HIGH RISK",C["amber"],"Annex III Pt.1: AUTOMATIC",
        "ALL biometric ID systems are automatically HIGH RISK. No opt-out unlike fraud detection. Full Arts.9-15 obligations: risk management, data governance, documentation, logging, transparency, human oversight, robustness.",
        "DEFINITELY ENGAGED — no exceptions"),
       ("LIMITED RISK",C["blue"],"Transparency obligations only","Not applicable.","NOT ENGAGED"),
       ("MINIMAL RISK",C["green"],"Largely unregulated","Not applicable.","NOT ENGAGED")]
cards="".join([
    f'<div style="flex:1;min-width:180px;background:rgba(255,255,255,.03);border:2px solid {c};border-radius:12px;padding:18px">'
    f'<div style="font-size:12px;font-weight:900;color:{c};font-family:monospace;text-transform:uppercase;margin-bottom:6px">{label}</div>'
    f'<div style="color:{C["text"]};font-size:12px;font-weight:700;margin-bottom:6px">{sub}</div>'
    f'<div style="color:{C["muted"]};font-size:11px;line-height:1.6;margin-bottom:10px">{desc}</div>'
    f'<div style="font-family:monospace;font-size:10px;color:{c};font-weight:700">{status}</div></div>'
    for label,c,sub,desc,status in tiers])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;padding:22px;font-family:sans-serif">'
    f'<div style="display:flex;flex-wrap:wrap;gap:12px;margin-bottom:18px">{cards}</div>'
    f'<div style="background:{C["panel2"]};border-radius:10px;padding:16px;border-left:4px solid {C["red"]}">'
    f'<div style="color:{C["red"]};font-weight:700;font-size:13px;margin-bottom:6px">&#9888; VERDICT: TWO-LAYER COMPLIANCE REQUIRED</div>'
    f'<div style="color:{C["text"]};font-size:12px;line-height:1.7">'
    f'The bank must: (a) separate visitor areas and switch off live matching or document a valid Art.5(1)(h) exception AND '
    f'(b) meet all Arts.9-15 obligations and register in the EU AI database before deployment. '
    f'Penalties up to <b>EUR 35M or 7% global turnover</b>.</div></div></div>'))

In [ ]:
# ── Module 3: Articles 9-15 Applied ─────────────────────────────────────────
articles=[
    ("Art.9","Risk Management",C["red"],"Risk tracked throughout lifecycle — accuracy drift, spoofing, emergency egress.",
     "Art.9(1): continuous iterative process throughout entire lifecycle","GAP","No lifecycle risk register; continuous process absent"),
    ("Art.10","Data Governance",C["orange"],"Training data must fairly represent all groups. NIST 2019: 100x higher errors for certain demographics.",
     "Art.10(1): data must be relevant, representative, free of errors","GAP","Training data not audited; proxy variable risk unmitigated"),
    ("Art.11","Technical Docs",C["orange"],"Full documentation before launch: design, thresholds, demographic accuracy, known weaknesses.",
     "Art.11: documentation drawn up before system placed on market","GAP","Vendor docs not shared with deployer bank"),
    ("Art.12","Record Keeping",C["amber"],"Every match, score, decision and override permanently logged. Employees have right to access own records.",
     "Art.12 + Art.14(5): logs must include identification of persons involved","PARTIAL","Logs exist but decision-path attribution not preserved"),
    ("Art.13","Transparency",C["red"],"Vendor publishes demographic accuracy. CNN explainability gap makes this unusually hard.",
     "Art.13(1): sufficiently transparent to enable deployers to interpret output","GAP","No XAI layer in production; no customer explanation mechanism"),
    ("Art.14","Human Oversight",C["red"],"SPECIAL BIOMETRIC RULE: TWO qualified humans must independently verify every match before action.",
     "Art.14(5): identification verified by at least two natural persons","GAP","No two-person verification protocol implemented — CRITICAL"),
    ("Art.15","Robustness",C["red"],"Accuracy documented across all demographic groups. Must resist fake photos, masks, deepfakes.",
     "Art.15(1): appropriate accuracy, robustness and cybersecurity throughout lifecycle","GAP","Drift monitoring absent; adversarial robustness untested"),
]
STATUS_COLS={"GAP":C["red"],"PARTIAL":C["amber"],"MET":C["green"]}
cards="".join([
    f'<div style="background:rgba(255,255,255,.02);border:1px solid {c}44;border-radius:12px;padding:16px;position:relative">'
    f'<div style="position:absolute;top:10px;right:10px;background:{STATUS_COLS[status]};color:white;'
    f'font-family:monospace;font-size:10px;font-weight:700;padding:2px 10px;border-radius:20px">{status}</div>'
    f'<div style="color:{c};font-size:17px;font-weight:900;font-family:monospace">{art}</div>'
    f'<div style="color:{C["text"]};font-weight:700;font-size:13px;margin:4px 0 8px">{name}</div>'
    f'<div style="color:{C["muted"]};font-size:11px;line-height:1.6;margin-bottom:8px">{desc}</div>'
    f'<div style="background:rgba(37,99,235,.08);border-left:3px solid {C["blue"]};padding:7px 10px;'
    f'border-radius:0 6px 6px 0;font-family:monospace;font-size:10px;color:{C["cyan"]};margin-bottom:6px">&#8594; {ref}</div>'
    f'<div style="font-family:monospace;font-size:10px;color:{STATUS_COLS[status]}">Gap: {gap}</div></div>'
    for art,name,c,desc,ref,status,gap in articles])
legend="".join([
    f'<span style="background:{v}22;border:1px solid {v}66;color:{v};padding:3px 12px;border-radius:20px;font-family:monospace;font-size:10px;margin-right:8px">&#9679; {k}</span>'
    for k,v in STATUS_COLS.items()])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;padding:22px;font-family:sans-serif">'
    f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(290px,1fr));gap:12px">{cards}</div>'
    f'<div style="display:flex;flex-wrap:wrap;margin-top:14px">{legend}</div></div>'))

In [ ]:
# ── Module 4: EU vs UK Comparison ───────────────────────────────────────────
rows_data=[
    ("Primary Instrument","Hard law binding on all 27 EU countries. Fully in force August 2026.",
     "Soft guidance only — no dedicated AI law. ICO + Surveillance Camera Commissioner. Deliberately flexible."),
    ("Live Biometric ID","Real-time facial recognition BANNED in public spaces — narrow law-enforcement exceptions only. Art.5(1)(h).",
     "Bridges v SWP [2020]: clear legal basis + completed DPIA + equality compliance. ICO guidance reinforces this."),
    ("Bias Testing","LAW REQUIRES bias testing and ongoing demographic accuracy checks throughout system lifetime.",
     "Equality Act + ICO guidance require bias to be addressed but no prescribed method. Clearview AI fined GBP 7.5m (2022)."),
    ("Human Oversight","TWO qualified humans must independently verify every biometric match before action. Art.14(5).",
     "ICO requires meaningful human review but NO equivalent two-person rule exists in UK law."),
    ("Penalties","Up to EUR 35M or 7% turnover (prohibited). EUR 15M or 3% (high-risk non-compliance).",
     "ICO: up to GBP 17.5M or 4% turnover. Bridges: judicial review; injunctive relief; unlimited HSE fines."),
    ("UK Bank Scope","UK banks with EU staff/visitors: if system affects EU persons, EU law applies.",
     "Applies within UK. Bridges binds UK courts. ICO guidance carries significant weight."),
]
rows_html="".join([
    f"<tr><td style='color:{C['cyan']};font-size:11px;font-family:monospace;padding:10px 12px;"
    f"border-bottom:1px solid rgba(37,99,235,.12);font-weight:700;white-space:nowrap'>{t}</td>"
    f"<td style='color:{C['text']};font-size:11px;padding:10px 12px;"
    f"border-bottom:1px solid rgba(37,99,235,.12);line-height:1.6'>{eu}</td>"
    f"<td style='color:{C['text']};font-size:11px;padding:10px 12px;"
    f"border-bottom:1px solid rgba(37,99,235,.12);line-height:1.6'>{uk}</td></tr>"
    for t,eu,uk in rows_data])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;overflow-x:auto;font-family:sans-serif">'
    f'<table style="width:100%;border-collapse:collapse;min-width:800px"><thead><tr>'
    f'<th style="background:{C["panel2"]};color:{C["muted"]};font-family:monospace;font-size:10px;letter-spacing:1px;padding:12px;text-align:left">DIMENSION</th>'
    f'<th style="background:rgba(239,68,68,.1);color:{C["red"]};font-family:monospace;font-size:11px;padding:12px;text-align:left">EU — EU AI Act + GDPR</th>'
    f'<th style="background:rgba(37,99,235,.1);color:{C["blue"]};font-family:monospace;font-size:11px;padding:12px;text-align:left">UK — Sectoral + Case Law</th>'
    f'</tr></thead><tbody>{rows_html}</tbody></table></div>'))

In [ ]:
# ── Module 5: Accountability Scenario ───────────────────────────────────────
parties=[
    (C["red"],"AI VENDOR","Biometric Provider",
     "Required to disclose accuracy by race and pregnancy-related changes. Warn system degrades as photos age. Failure to register in EU AI database before launch.",
     "EU AI Act Art.13: demographic accuracy required | Art.15: declared in instructions | Art.49: register before deployment EUR 15M/3%"),
    (C["blue"],"DEPLOYER","The Bank",
     "Data controller + responsible person under fire safety law. Doors should have unlocked on alarm. Indirect discrimination likely. Named Senior Manager personally accountable.",
     "RRO 2005 Art.4: routes to safety clear | Equality Act s.19(1): indirect discrimination | Bridges [2020]: current DPIA required"),
    (C["amber"],"REGULATORS","ICO / HSE / Tribunal",
     "ICO investigates DPIA adequacy. HSE: unlimited fines + criminal prosecution of Senior Manager. Employment tribunal: uncapped damages for indirect discrimination.",
     "UK GDPR Art.9: biometric processing requires valid basis | Equality Act s.29: providers must not discriminate | RRO 2005 Art.32: unlimited fines"),
]
scenario=(
    f'<div style="background:rgba(239,68,68,.08);border:1px solid rgba(239,68,68,.4);'
    f'border-radius:10px;padding:14px 18px;margin-bottom:18px">'
    f'<div style="color:{C["red"]};font-family:monospace;font-size:10px;letter-spacing:2px;margin-bottom:6px">SCENARIO</div>'
    f'<div style="color:{C["text"]};font-size:12px;line-height:1.7">'
    f'Fire alarm goes off. System fails to recognise a Black woman six months pregnant — face changed since enrolment, '
    f'score 78% (below threshold). Sent to manual review but officer is swamped. She waits 4+ minutes, panics, falls and miscarries. '
    f'Later testing shows system rejects Black women at <b>4x the rate</b> of white men.</div></div>')
cols_html="".join([
    f'<div style="background:{C["panel2"]};border-top:4px solid {c};border-radius:10px;padding:16px">'
    f'<div style="color:{c};font-weight:900;font-size:13px;margin-bottom:3px">{party}</div>'
    f'<div style="color:{C["muted"]};font-size:10px;margin-bottom:10px">{role}</div>'
    f'<div style="color:{C["text"]};font-size:11px;line-height:1.7;margin-bottom:8px">{body}</div>'
    f'<div style="font-family:monospace;font-size:10px;color:{c};line-height:1.6">{legal}</div></div>'
    for c,party,role,body,legal in parties])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;padding:22px;font-family:sans-serif">'
    f'{scenario}'
    f'<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px">{cols_html}</div></div>'))

In [ ]:
# ── Module 6: Human-in-the-Loop ─────────────────────────────────────────────
hitl_steps=[
    (C["blue"],"1","Face Captured","Cameras with depth anti-spoofing. Rejects 2D photos. Full process under 400ms."),
    (C["cyan"],"2","Match Score Computed","128-D embedding vs all enrolled staff. Confidence 0-100% returned with 3 closest matches."),
    (C["purple"],"3","Threshold Routing","Above 92%: door opens. 80-92%: reception second-check. Below 80%: security officer full review."),
    (C["amber"],"4","Human Officer Decision","Officer compares live frame to ID badge, asks questions, records written rationale."),
    (C["green"],"5","Audit and Feedback","Every decision permanently logged. Overrides feed quarterly model updates."),
]
failures=[
    (C["red"],"Guard Deskilling","Officers become button-pressers. Art.14(5) two-person rule ensures human present — not thinking critically. Floor, not ceiling."),
    (C["orange"],"Queue Pressure","At 08:00 Monday, officers face wall of impatient employees. Path of least resistance: wave through borderline scores."),
    (C["amber"],"Computer Says No Effect","Guards less likely to override AI rejections even with valid ID present. Amplifies demographic bias for affected groups."),
]
flow="".join([
    f'<div style="display:flex;gap:12px;align-items:flex-start;background:rgba(255,255,255,.02);'
    f'border:1px solid {c}33;border-radius:8px;padding:10px 14px">'
    f'<div style="width:26px;height:26px;border-radius:50%;background:{c};color:white;font-weight:900;'
    f'font-size:12px;font-family:monospace;flex-shrink:0;display:flex;align-items:center;justify-content:center">{n}</div>'
    f'<div><div style="color:{c};font-weight:700;font-size:12px">{t}</div>'
    f'<div style="color:{C["muted"]};font-size:11px;margin-top:2px">{d}</div></div></div>'
    + (f'<div style="text-align:center;color:{C["muted"]};font-size:18px;margin:2px 0">&#8595;</div>' if i<4 else "")
    for i,(c,n,t,d) in enumerate(hitl_steps)])
fail_html="".join([
    f'<div style="border-left:4px solid {c};padding:10px 14px;margin-bottom:8px;background:rgba(255,255,255,.02);border-radius:0 8px 8px 0">'
    f'<div style="color:{c};font-weight:700;font-size:12px;margin-bottom:4px">{t}</div>'
    f'<div style="color:{C["muted"]};font-size:11px;line-height:1.6">{d}</div></div>'
    for c,t,d in failures])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;padding:22px;font-family:sans-serif">'
    f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:18px">'
    f'<div><div style="color:{C["cyan"]};font-family:monospace;font-size:10px;letter-spacing:2px;margin-bottom:12px">HUMAN-IN-THE-LOOP DESIGN</div>{flow}</div>'
    f'<div><div style="color:{C["red"]};font-family:monospace;font-size:10px;letter-spacing:2px;margin-bottom:12px">THREE WAYS AI SHIFTS GUARD BEHAVIOUR</div>'
    f'{fail_html}'
    f'<div style="background:rgba(37,99,235,.08);border:1px solid rgba(37,99,235,.25);border-radius:8px;padding:12px;margin-top:12px">'
    f'<div style="color:{C["blue"]};font-family:monospace;font-size:10px;letter-spacing:2px;margin-bottom:6px">ART.14(5) TWO-PERSON RULE</div>'
    f'<div style="color:{C["text"]};font-size:11px;line-height:1.7">For biometric ID: no action is taken unless identification has been separately verified and confirmed by at least two natural persons. This is a <b>floor</b> — not a guarantee of genuine critical thinking.</div>'
    f'</div></div></div></div>'))

In [ ]:
# ── Module 7: Stakeholder Map ────────────────────────────────────────────────
stakeholders=[
    ("Employees\n(Data Subjects)",8,9,C["blue"],50,
     "Role: Subject of biometric ID<br>Interest: Reliable non-discriminatory authentication; safety<br>Risk: Repeated false rejections; physical safety; template compromise"),
    ("Visitors\n(Data Subjects)",5,7,C["cyan"],40,
     "Role: Subject of biometric ID in lobby<br>Interest: Clear notification; lawful basis<br>Risk: Scanning visitors in lobby may be ILLEGAL under Art.5(1)(h)"),
    ("Security Officers",6,6,C["purple"],42,
     "Role: Human oversight operators<br>Interest: Tools that support judgement not replace it<br>Risk: Deskilling; queue pressure; over-trusting wrong AI rejections"),
    ("AI/Biometric Vendor",7,8,C["amber"],48,
     "Role: System provider<br>Interest: Effective deployment; manage AI Act product liability<br>Risk: High-risk classification triggers expensive compliance; liability if bias undisclosed"),
    ("ICO",9,7,C["red"],44,
     "Role: Data protection regulator<br>Interest: UK GDPR compliance; complete DPIA per Bridges<br>Risk: Sector-wide non-compliance; processing without valid basis"),
    ("HSE / Building Safety",8,8,C["orange"],46,
     "Role: Workplace and fire safety regulator<br>Interest: Access controls do not impede emergency egress<br>Risk: Door locks in fire traps people; inadequate fire alarm integration"),
    ("EHRC",9,6,C["green"],40,
     "Role: Equality regulator<br>Interest: Non-discriminatory deployment; reasonable adjustments<br>Risk: Disproportionate error rates = indirect discrimination; Equality Act s.20"),
    ("Board / SM&CR\nSenior Manager",10,10,C["red"],56,
     "Role: Ultimate accountability holder<br>Interest: Regulatory + reputational + financial risk<br>Risk: Personal liability; reputational damage; criminal prosecution for fire safety"),
]
fig=go.Figure()
fig.add_trace(go.Scatter(
    x=[s[2] for s in stakeholders],y=[s[1] for s in stakeholders],
    mode="markers+text",
    marker=dict(size=[s[4] for s in stakeholders],color=[s[3] for s in stakeholders],
                line=dict(color="#050B14",width=2),opacity=0.88),
    text=[s[0].replace("\n"," ") for s in stakeholders],textposition="top center",
    textfont=dict(color="#E8F0FE",size=10,family="monospace"),
    hovertext=["<b>"+s[0].replace("\n"," ")+"</b><br><br>"+s[5] for s in stakeholders],
    hoverinfo="text"))
fig.add_vline(x=7.5,line=dict(color="rgba(37,99,235,.25)",dash="dot",width=1.5))
fig.add_hline(y=7.5,line=dict(color="rgba(37,99,235,.25)",dash="dot",width=1.5))
layout=dark_layout("Stakeholder Map — Power vs Impact (hover for full detail)",h=500)
layout.update(xaxis=dict(title="Impact",range=[3,12],tickfont=dict(color=C["muted"]),gridcolor="rgba(37,99,235,.1)",zeroline=False),
              yaxis=dict(title="Power / Influence",range=[3,12],tickfont=dict(color=C["muted"]),gridcolor="rgba(37,99,235,.1)",zeroline=False),
              showlegend=False)
fig.update_layout(**layout); fig.show()

In [ ]:
# ── Module 8: Risk Register RAG + L x I ─────────────────────────────────────
z_matrix=[[l*i for i in range(1,6)] for l in range(1,6)]
colorscale=[[0,"#0F2040"],[0.2,"#0F2040"],[0.45,"#2D2A10"],[0.65,"#7C3A0A"],[0.85,"#991A1A"],[1,C["red"]]]
fig=make_subplots(rows=1,cols=2,column_widths=[0.62,0.38],
    subplot_titles=["L x I Risk Matrix","Risk by Severity"],
    specs=[[{"type":"heatmap"},{"type":"bar"}]])
fig.add_trace(go.Heatmap(z=z_matrix,x=[1,2,3,4,5],y=[1,2,3,4,5],
    colorscale=colorscale,showscale=True,zmin=1,zmax=25,opacity=0.85,
    colorbar=dict(title="LxI",tickfont=dict(color=C["muted"])),
    hovertemplate="L:%{y} x I:%{x} = %{z}<extra></extra>"),row=1,col=1)
for l in range(1,6):
    for i in range(1,6):
        fig.add_annotation(x=i,y=l,text=str(l*i),font=dict(size=9,color="rgba(255,255,255,.2)"),showarrow=False,row=1,col=1)
for _,row in RISKS.iterrows():
    fig.add_trace(go.Scatter(x=[row["I"]],y=[row["L"]],mode="markers+text",
        marker=dict(size=36,color=row["SevCol"],line=dict(color=C["bg"],width=2),opacity=0.92),
        text=[row["ID"].replace("R-","")],textfont=dict(size=9,color="white"),
        textposition="middle center",name=row["ID"],
        hovertemplate=f"<b>{row['ID']}</b><br>{row['Risk'][:50]}<br>L={row['L']} x I={row['I']}={row['Score']} ({row['Severity']})<extra></extra>"),row=1,col=1)
sev_order=["Critical","High","Medium","Low"]
counts=[RISKS[RISKS["Severity"]==s].shape[0] for s in sev_order]
fig.add_trace(go.Bar(x=sev_order,y=counts,
    marker=dict(color=[SEV_COL[s] for s in sev_order],line=dict(color=C["bg"],width=2)),
    text=counts,textposition="outside",textfont=dict(color="white",size=13),
    hovertemplate="%{x}: %{y} risks<extra></extra>"),row=1,col=2)
layout=dark_layout("Risk Register — L x I Scoring with Regulatory Traceability",h=520)
layout.update(xaxis=dict(title="Impact",tickvals=[1,2,3,4,5],gridcolor="rgba(37,99,235,.1)",tickfont=dict(color=C["muted"])),
              yaxis=dict(title="Likelihood",tickvals=[1,2,3,4,5],gridcolor="rgba(37,99,235,.1)",tickfont=dict(color=C["muted"])),
              xaxis2=dict(gridcolor="rgba(37,99,235,.1)",tickfont=dict(color=C["muted"])),
              yaxis2=dict(title="Count",gridcolor="rgba(37,99,235,.1)",tickfont=dict(color=C["muted"])),showlegend=False)
fig.update_layout(**layout); fig.update_annotations(font=dict(color=C["muted"],size=10)); fig.show()
crit=RISKS[RISKS["Severity"]=="Critical"]
print(f"Critical risks (Score>=16): {len(crit)}")
for _,r in crit.iterrows(): print(f"  {r['ID']}  L={r['L']}xI={r['I']}={r['Score']}  {r['Risk'][:60]}")

In [ ]:
# ── Module 9: Compliance Knowledge Graph (pyvis) ─────────────────────────────
G=nx.DiGraph()
for aid in ARTICLES: G.add_node(aid)
for src,tgt,*_ in EDGES: G.add_edge(src,tgt)
centrality=nx.degree_centrality(G)
net=Network(height="700px",width="100%",bgcolor=C["bg"],font_color=C["text"],
            directed=True,notebook=True,cdn_resources="remote")
net.set_options('{"nodes":{"font":{"size":11,"face":"Courier New","color":"#E8F0FE"},"borderWidth":2,"shadow":{"enabled":true,"size":12}},'
                '"edges":{"arrows":{"to":{"enabled":true,"scaleFactor":0.7}},"color":{"inherit":false},'
                '"smooth":{"type":"curvedCW","roundness":0.2},"font":{"size":8,"face":"monospace","color":"#94A3B8","strokeWidth":0}},'
                '"physics":{"barnesHut":{"gravitationalConstant":-10000,"springLength":200,"springConstant":0.03},"stabilization":{"iterations":300}},'
                '"interaction":{"hover":true,"tooltipDelay":60,"zoomView":true,"dragView":true}}')
for aid,art in ARTICLES.items():
    col=STATUS_COL[art["status"]]; sz=(52 if art["status"]=="TRIGGER" else 34)+centrality.get(aid,0)*90
    tip=(f"<b style='color:{col}'>{art['label']}</b><br><b>Requires:</b> {art['what']}<br><br>"
         f"<b>Status:</b> <span style='color:{col}'>{art['status']}</span><br><b>Gap:</b> {art['gap']}")
    net.add_node(aid,label=art["label"],color=col,size=sz,title=tip,borderWidth=2,shadow=True)
for src,tgt,lbl,col,w in EDGES:
    net.add_edge(src,tgt,label=lbl,color={"color":col,"opacity":0.78},width=w,
                 title=f"<b>{lbl}</b><br>{src} to {tgt}")
net.save_graph("compliance_graph.html")
display(HTML(open("compliance_graph.html").read()))
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.4);border-radius:12px;padding:14px 20px;margin-top:8px;font-family:monospace;font-size:11px">'
    f'<span style="color:{C["muted"]};letter-spacing:2px;font-size:10px">NODE LEGEND </span> '
    f'<span style="color:{C["text"]}"><span style="color:{C["blue"]}">&#9679;</span> TRIGGER</span> &nbsp;'
    f'<span style="color:{C["text"]}"><span style="color:{C["red"]}">&#9679;</span> GAP({gaps_n})</span> &nbsp;'
    f'<span style="color:{C["text"]}"><span style="color:{C["amber"]}">&#9679;</span> PARTIAL({partial_n})</span> &nbsp;'
    f'<span style="color:{C["text"]}"><span style="color:{C["green"]}">&#9679;</span> MET({met_n})</span><br>'
    f'<span style="color:{C["muted"]};font-size:10px">Drag nodes · Hover for gap detail · Scroll to zoom · Edge label = relationship type</span></div>'))
top5=sorted(centrality.items(),key=lambda x:x[1],reverse=True)[:5]
print("Most central articles (highest governance risk):")
for aid,sc in top5: print(f"  {ARTICLES[aid]['label']:<28}  centrality={sc:.3f}  status={ARTICLES[aid]['status']}")

In [ ]:
# ── Module 10: Reflective Commentary ────────────────────────────────────────
reflections=[
    ("&#9878;",C["red"],"Lab Accuracy vs Lobby Reality",
     "I started this module assuming facial recognition was basically a solved problem — vendors advertise 99.9% accuracy and the tech is everywhere. Reading the NIST FRVT 2019 data and Gender Shades genuinely changed that. Those headline numbers fall apart when you break them down by race, gender or age. A system that performs brilliantly in the lab can quietly fail the very people who do not look like the development team's training data. The real question is not how accurate is it — it is accurate for whom, and where does it go wrong?"),
    ("&#9878;",C["blue"],"Bridges as the Turning Point",
     "Reading the Court of Appeal judgment in Bridges v SWP [2020] EWCA Civ 1058 was the moment the legal dimension became real. The Court did not say facial recognition is unlawful — it said you need a clear legal framework, a completed DPIA, and equality compliance, and South Wales Police had none of those. I used to think legal compliance was something bolted on after the technical design was done. Bridges made clear it must come first. That judgment binds UK courts and carries weight well beyond policing."),
    ("&#128295;",C["amber"],"Cross-Cutting Accountability",
     "The biggest shift for me is realising that biometric systems do not sit neatly inside any single area of law. The fire evacuation scenario touches data protection, equality, employment, fire safety, and AI law all at once — five separate regulators, five separate duties. A cyber security professional who only thinks in one of those areas will miss serious risks. I now see legal literacy across all these domains not as a soft skill but as core technical competence."),
]
ref_html="".join([
    f'<div style="display:flex;gap:14px;border-left:4px solid {c};padding:14px 18px;margin-bottom:12px;background:rgba(255,255,255,.02);border-radius:0 10px 10px 0">'
    f'<div style="font-size:26px;flex-shrink:0">{icon}</div>'
    f'<div><div style="color:{c};font-weight:700;font-size:13px;margin-bottom:6px">{title}</div>'
    f'<div style="color:{C["muted"]};font-size:12px;line-height:1.8">{body}</div></div></div>'
    for icon,c,title,body in reflections])
display(HTML(
    f'<div style="background:{C["panel"]};border:1px solid rgba(37,99,235,.35);border-radius:14px;padding:26px;font-family:sans-serif">{ref_html}</div>'))

In [ ]:
# ── Module 11: Executive Dashboard ──────────────────────────────────────────
eu_score=round(met_n/max(len(non_trigger),1)*100)
ukwp_score=20
risk_score=round((1-RISKS["Score"].sum()/(len(RISKS)*25))*100)
crit_n=int((RISKS["Score"]>=16).sum())

fig=make_subplots(rows=2,cols=3,
    specs=[[{"type":"indicator"},{"type":"indicator"},{"type":"indicator"}],
           [{"type":"bar","colspan":3},None,None]],
    subplot_titles=["EU AI Act Compliance","UK AI White Paper","Risk Posture",
                    "Regulatory Obligation Priorities & Deadlines"],
    row_heights=[0.42,0.58],vertical_spacing=0.12)
for ci,(sc,ttl,col) in enumerate([(eu_score,"EU AI Act",C["red"]),(ukwp_score,"UK AI WP",C["amber"]),(risk_score,"Risk Posture",C["green"])],1):
    fig.add_trace(go.Indicator(mode="gauge+number+delta",value=sc,
        delta={"reference":80,"valueformat":".0f","font":{"size":11,"color":C["muted"]}},
        gauge=dict(axis=dict(range=[0,100],tickfont=dict(color=C["muted"],size=9)),
                   bar=dict(color=col,thickness=0.3),bgcolor=C["panel"],borderwidth=1,
                   bordercolor="rgba(37,99,235,.3)",
                   steps=[dict(range=[0,50],color="rgba(239,68,68,.08)"),
                          dict(range=[50,80],color="rgba(245,158,11,.08)"),
                          dict(range=[80,100],color="rgba(34,197,94,.08)")],
                   threshold=dict(line=dict(color=C["cyan"],width=3),thickness=0.75,value=80)),
        number=dict(font=dict(color="white",size=34),suffix="%"),
        title=dict(text=f"<b>{ttl}</b>",font=dict(color=C["muted"],size=11))),row=1,col=ci)
obligs=[("EU AI Act Arts.9-17",3,C["red"],"Aug 2026"),("UK GDPR Art.9",3,C["red"],"In force"),
        ("UK GDPR Art.22",3,C["red"],"In force"),("FCA Consumer Duty",3,C["red"],"Since Jul 23"),
        ("Equality Act s.19",3,C["red"],"In force"),("Bridges DPIA Req.",3,C["red"],"In force"),
        ("RRO 2005 Fire",3,C["red"],"In force"),("Art.5 Prohibition",3,C["red"],"Aug 2026"),
        ("NIS2 Cybersecurity",2,C["amber"],"In force")]
fig.add_trace(go.Bar(x=[o[0] for o in obligs],y=[o[1] for o in obligs],
    marker=dict(color=[o[2] for o in obligs],line=dict(color="#050B14",width=2)),
    text=[f"Deadline:{o[3]}" for o in obligs],textposition="outside",
    textfont=dict(color=C["muted"],family="monospace",size=8),
    hovertemplate="%{x}<br>Priority:%{y}/3<br>%{text}<extra></extra>",width=0.55),row=2,col=1)
layout=dark_layout("Executive Compliance Dashboard — Board-Level Reporting",h=660)
layout.update(xaxis2=dict(tickfont=dict(color=C["muted"],family="monospace",size=8),tickangle=-30,gridcolor="rgba(37,99,235,.1)"),
              yaxis2=dict(tickvals=[1,2,3],ticktext=["Medium","High","Critical"],tickfont=dict(color=C["muted"]),gridcolor="rgba(37,99,235,.1)"),
              showlegend=False)
fig.update_layout(**layout); fig.update_annotations(font=dict(color=C["muted"],size=10)); fig.show()

kpis=[("HIGH RISK","EU AI Act Classification","Annex III Pt.1 + Art.5 potential ban",C["red"],"rgba(239,68,68,.12)","rgba(239,68,68,.4)"),
      (str(crit_n),"Critical Risks (>=16)","Immediate board action required",C["red"],"rgba(239,68,68,.12)","rgba(239,68,68,.4)"),
      (f"{gaps_n}/{len(non_trigger)}","Article Gaps","All primary obligations have gaps",C["red"],"rgba(239,68,68,.12)","rgba(239,68,68,.4)"),
      (f"4x","Higher Error Rate","Black women vs white men (NIST 2019)",C["orange"],"rgba(249,115,22,.12)","rgba(249,115,22,.4)"),
      (f"GBP 17.5M+","Max ICO Penalty","4% global turnover exposure",C["red"],"rgba(239,68,68,.12)","rgba(239,68,68,.4)"),
      ("Unlimited","HSE Fire Safety","Fines and imprisonment possible",C["red"],"rgba(239,68,68,.12)","rgba(239,68,68,.4)")]
cards_html="".join([
    f'<div style="background:{bg};border:1px solid {brd};border-radius:10px;padding:14px">'
    f'<div style="font-size:22px;font-weight:800;color:{col}">{n}</div>'
    f'<div style="color:#E8F0FE;font-size:11px;font-weight:600;margin-top:4px">{lbl}</div>'
    f'<div style="color:#64748B;font-size:9px;margin-top:2px">{sub}</div></div>'
    for n,lbl,sub,col,bg,brd in kpis])
p1=["Commission independent ML bias audit (Equality Act s.19 + Art.10)",
    "Separate visitor areas — switch off live matching or document Art.5(1)(h) exception",
    "Renegotiate vendor contract: XAI + audit rights + Art.16 compliance verification",
    "Implement Art.14(5) two-person verification protocol with written rationale logging",
    "Test and document fire alarm door-release integration under RRO 2005",
    "Complete DPIA per Bridges judgment covering all demographic accuracy data"]
actions_html="".join([
    f'<div style="border-left:3px solid {C["blue"]};padding:9px 14px;margin-bottom:6px;'
    f'background:rgba(37,99,235,.07);border-radius:0 8px 8px 0;color:{C["text"]};font-size:11px">&#9658; {a}</div>'
    for a in p1])
display(HTML(
    f'<div style="background:linear-gradient(135deg,{C["bg"]},{C["panel2"]});border:1px solid rgba(37,99,235,.3);'
    f'border-radius:16px;padding:28px;margin-top:12px;font-family:Courier New,monospace">'
    f'<div style="color:{C["cyan"]};letter-spacing:3px;font-size:10px;margin-bottom:14px">EXECUTIVE SUMMARY · {datetime.now().strftime("%d %B %Y %H:%M")}</div>'
    f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:10px;margin-bottom:20px">{cards_html}</div>'
    f'<div style="color:{C["muted"]};font-size:10px;margin-bottom:8px;text-transform:uppercase;letter-spacing:1px">P1 Immediate Actions</div>'
    f'{actions_html}'
    f'<div style="margin-top:14px;color:{C["muted"]};font-size:9px;border-top:1px solid rgba(37,99,235,.15);padding-top:10px">'
    f'7CS525 Human &amp; Legal Aspects of Cyber Security &middot; University of Derby 2025/26 &middot; Student: 100796311</div></div>'))
print(f"\n{'='*60}\n  ALL 11 MODULES COMPLETE\n  EU AI Act: {eu_score}%  |  UK AI WP: {ukwp_score}%\n  Critical risks: {crit_n}  |  Article gaps: {gaps_n}/{len(non_trigger)}\n{'='*60}")